# Lesson 08 — Linear Algebra Intuition\n
**What this notebook does:** turns a support ticket into a *vector* (a list of numbers), turns a batch of tickets into a *matrix* (a table of those lists), and uses the *dot product* to compute an urgency-style score for one ticket and then for many tickets at once. Every idea here is built with plain Python first, then with NumPy, so you can see that NumPy is not doing anything mysterious — it is doing the exact same arithmetic, just faster and with less typing.

## A ticket as a vector
A **vector** is just an ordered list of numbers, where each position (called a **component**) means something specific. Take one support ticket and describe it with three numbers: how many words it has, how many exclamation marks it has, and how many words from an "urgent word" list it contains (words like *urgent*, *now*, *immediately*). That list of three numbers, in that order, is a vector.
We build it below as a plain Python list first, then as a NumPy array — the same numbers, two different containers."

In [70]:
import numpy as np
ticket_text = "My payment failed twice, I need this resolved right now!!"

# [word_count, exclamation_count, urgent_word_count]
ticket_as_list = [10, 2, 2]
ticket_as_vector = np.array(ticket_as_list)

print(ticket_as_list)
print(ticket_as_vector)
print(type(ticket_as_list), type(ticket_as_vector))

[10, 2, 2]
[10  2  2]
<class 'list'> <class 'numpy.ndarray'>


## Vector arithmetic: add and scale
Two things you can do to vectors: **add** two vectors together (add matching positions), and **scale** a vector by a number (multiply every position by that number). Both are done position-by-position — the first number only ever interacts with the first number, the second only with the second, and so on.
Say a second ticket arrives with vector `[4, 0, 1]`. Adding the two vectors gives a combined total; scaling `ticket_as_vector` by `2` doubles every component, as if the same ticket's signals were twice as strong."

In [71]:
second_ticket_vector = np.array([4, 0, 1])
combined = ticket_as_vector + second_ticket_vector
doubled = ticket_as_vector * 2
print("combined:", combined)
print("doubled: ", doubled)

combined: [14  2  3]
doubled:  [20  4  4]


## The dot product: turning a vector into one number
The **dot product** of two same-length vectors is: multiply each matching pair of numbers, then add up all those products. It takes two lists of numbers and collapses them into a single number.\n\nGive each of our three ticket signals a **weight** — how much it should count toward an urgency score: word count matters a little (weight `0.1`), each exclamation mark matters more (weight `1.5`), and each urgent word matters most (weight `3`). The weight vector is `[0.1, 1.5, 3]`.\n\nFor `ticket_as_vector = [10, 2, 2]`, the dot product with the weights is:\n\n```\n(10 × 0.1) + (2 × 1.5) + (2 × 3)\n  =  1.0   +   3.0     +   6.0\n  =  10.0\n```\n\nThat single number, `10.0`, is an urgency score built entirely from a dot product. This is exactly the same shape of calculation as Lesson 1's word-counting classifier — a set of signals, each multiplied by how much it matters, then summed. We compute it three ways below: a hand-written loop, NumPy's `np.dot`, and the `@` operator, to show all three give the identical answer."

In [72]:
weights = np.array([0.1, 1.5, 3])
# hand-written loop version\n
manual_total = 0
for signal, weight in zip(ticket_as_vector, weights):
    manual_total += signal * weight
# NumPy versions
dot_version = np.dot(ticket_as_vector, weights)
at_version = ticket_as_vector @ weights
print("loop result:", manual_total)
print("np.dot result:", dot_version)
print("@ result:", at_version)

loop result: 10.0
np.dot result: 10.0
@ result: 10.0


## A matrix: many tickets, stacked into a table
A **matrix** is a grid of numbers arranged in rows and columns. If a vector is one shopping list, a matrix is a whole spreadsheet of shopping lists — one row per list, one column per item. Here, each **row** is one ticket's vector `[word_count, exclamation_count, urgent_word_count]`, and each **column** is one signal, repeated down every ticket.Four tickets stacked this way form a 4-row, 3-column matrix — written as **shape `(4, 3)`**: 4 rows, 3 columns, in that order.

In [73]:
tickets_matrix = np.array([[10, 2, 2],   
                           # "My payment failed twice, I need this resolved right now!!"
                           [4, 0, 1],    
                           # "urgent: order missing"
                           [8, 0, 0],    
                           #"Do you have this item in a larger size"
                           [15, 3, 3],   
                           #"This is unacceptable, I need a refund immediately!!! now!!"
                           ])
print(tickets_matrix)
print("shape:", tickets_matrix.shape)

[[10  2  2]
 [ 4  0  1]
 [ 8  0  0]
 [15  3  3]]
shape: (4, 3)


## Matrix times vector: scoring every ticket at once
Here is the payoff. Instead of computing one dot product per ticket by hand, multiply the whole **matrix** by the **weight vector** in one shot: `tickets_matrix @ weights`. NumPy takes each row of the matrix, dot-products it with `weights`, and hands back one score per row — four tickets in, four scores out, no loop written by us.
For this to work, the number of **columns** in the matrix must match the number of entries in the vector — 3 signals per ticket, 3 weights. That matching-length rule is the one rule to remember about matrix-vector multiplication; the next cell after this one shows what happens when it's broken."

In [74]:
scores = tickets_matrix @ weights
for row, score in zip(tickets_matrix, scores):
    print(f"{row} -> score {score}")
    print("matrix shape:", tickets_matrix.shape, "weights shape:", weights.shape, "scores shape:", scores.shape)

[10  2  2] -> score 10.0
matrix shape: (4, 3) weights shape: (3,) scores shape: (4,)
[4 0 1] -> score 3.4
matrix shape: (4, 3) weights shape: (3,) scores shape: (4,)
[8 0 0] -> score 0.8
matrix shape: (4, 3) weights shape: (3,) scores shape: (4,)
[15  3  3] -> score 15.0
matrix shape: (4, 3) weights shape: (3,) scores shape: (4,)


## Shapes must match — what happens when they don't
If the weight vector has the wrong number of entries — say only 2 weights for 3-column tickets — NumPy refuses and raises an error rather than silently computing something wrong. Seeing the real error message once makes the shape rule much easier to remember than being told about it."

In [75]:
wrong_length_weights = np.array([0.1, 1.5])  # only 2 numbers, tickets_matrix has 3 columns
tickets_matrix @ wrong_length_weights

ValueError: matmul: Input operand 1 has a mismatch in its core dimension 0, with gufunc signature (n?,k),(k,m?)->(n?,m?) (size 2 is different from 3)